# 07.1 — Carga e Feature Engineering dos Recém-Nascidos

**Purpose:** Load both data sources (Oracle RECEM_NASCIDO + DuckDB RN atendimentos),
merge with maternal data, apply feature engineering, generate Table 1, and save
shared artifacts to `out/artifacts_rn/` for downstream notebooks.

Upstream dependency: `01_eda_gestante.py` must have run first so that
`gestante_enriquecido` exists in `data/gestante.duckdb`.

## 1. Setup

In [ ]:
from __future__ import annotations
import sys, warnings, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import duckdb

warnings.filterwarnings("ignore")

ROOT     = Path(__file__).resolve().parent.parent
PARQUET  = ROOT / "data" / "gestante_quadro_clinico.parquet"
RN_PARQUET = ROOT / "data" / "recem_nascido_oracle.parquet"
OUT_DIR  = ROOT / "out" / "07.1_carga_rn"
ART_DIR  = ROOT / "out" / "artifacts_rn"

OUT_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "src"))

# ── Idioma / Language ─────────────────────────────────────────────────────────
LANG = "pt"  # "pt" | "en"  ← única linha a alterar para mudar idioma
# (este arquivo não gera figuras — flag mantida por consistência com os demais)

print(f"ROOT      : {ROOT}")
print(f"PARQUET   : {PARQUET.exists()}")
print(f"RN Oracle : {RN_PARQUET}  existe={RN_PARQUET.exists()}")

## 2. Export RN_PARQUET if missing

In [ ]:
if not RN_PARQUET.exists():
    print("\n[07.1] Exportando RECEM_NASCIDO do Oracle (necessario uma vez)...")
    import subprocess
    subprocess.run([sys.executable, str(ROOT / "src" / "export_recem_nascido.py")], check=True)

## 3. Load Fonte 1: RECEM_NASCIDO Oracle

In [ ]:
df_rn_oracle = pd.read_parquet(str(RN_PARQUET))
df_rn_oracle.columns = df_rn_oracle.columns.str.lower()
print(f"[Fonte 1] RECEM_NASCIDO Oracle: {len(df_rn_oracle):,} bebes")

## 4. Load Fonte 2: RN atendimentos (DuckDB)

In [ ]:
pq_abs = str(PARQUET.resolve())
con = duckdb.connect(str(ROOT / "data" / "gestante.duckdb"))
con.execute(
    f"CREATE OR REPLACE VIEW recem_nascido_atendimento AS "
    f"SELECT * FROM read_parquet('{pq_abs}') WHERE is_recem_nascido=1"
)
con.execute(
    f"CREATE OR REPLACE VIEW gestante AS "
    f"SELECT * FROM read_parquet('{pq_abs}') WHERE is_recem_nascido=0"
)

df_rn_atend = con.execute("""
    SELECT
        r.cd_atendimento       AS cd_atend_rn,
        r.cd_paciente          AS cd_paciente_rn,
        r.cd_atendimento_pai   AS cd_atend_mae,
        r.cd_cid               AS cid_rn,
        r.sn_obito             AS obito_rn,
        r.dt_atendimento       AS dt_internacao_rn,
        r.dt_alta              AS dt_alta_rn
    FROM recem_nascido_atendimento r
""").df()
df_rn_atend.columns = df_rn_atend.columns.str.lower()
print(f"[Fonte 2] Atendimentos RN     : {len(df_rn_atend):,}")
print(f"          Obitos (internacao) : {(df_rn_atend['obito_rn'].fillna('N')=='S').sum():,}")

## 5. Load maternal data (DuckDB)

In [ ]:
df_mae_raw = con.execute("""
    SELECT
        cd_atendimento                       AS cd_atend_mae_key,
        cd_cid                               AS cid_mae,
        nm_paciente                          AS nm_mae,
        CAST(idade_anos AS DOUBLE)           AS idade_mae,
        ds_cid_principal                     AS ds_cid_mae,
        CAST(semanas_gestacao AS DOUBLE)     AS ig_mae_struct,
        sn_gestacao_tiss,
        sn_cesario_tiss,
        sn_complicacao_tiss,
        CAST(vl_escore_triagem AS DOUBLE)    AS escore_triagem_mae,
        CAST(qtd_evolucoes_documentadas AS DOUBLE) AS evol_mae,
        ds_queixa_principal,
        txt_sinais_sintomas,
        txt_evolucao_medica,
        txt_just_internacao,
        txt_obs_gestacao,
        txt_intercorrencias
    FROM gestante
""").df()
con.close()
df_mae_raw.columns = df_mae_raw.columns.str.lower()
print(f"[Maes]    Gestantes           : {len(df_mae_raw):,}")

## 6. Load maternal enriched features (from gestante_enriquecido)

Loads pre-computed obstetric features from DuckDB instead of re-running
enrich_obstetric (which is slow and already done by 01_eda_gestante.py).

In [ ]:
_con = duckdb.connect(str(ROOT / "data" / "gestante.duckdb"))
_tabs = _con.execute("SHOW TABLES").df()["name"].tolist()

if "gestante_enriquecido" in _tabs:
    df_mae_enr = _con.execute("""
        SELECT cd_atendimento AS cd_atend_mae_key,
               G_obs, P_obs, C_obs, A_obs, N_obs, IG_obs,
               tipo_gestacao, perfil_historico, risco_gestacao,
               gestacao_gemelar, gestacao_pretermo, primigesta,
               grande_multipara_v2, tem_aborto_v2, aborto_recorrente,
               sem_filhos_vivos, perda_previa, cd_cid
        FROM gestante_enriquecido
    """).df()
    _con.close()
    df_mae_enr.columns = df_mae_enr.columns.str.lower()
    # Restore mixed-case enriched columns
    _MIXED = ["G_obs", "P_obs", "C_obs", "A_obs", "N_obs", "IG_obs",
              "tipo_gestacao", "perfil_historico", "risco_gestacao",
              "gestacao_gemelar", "gestacao_pretermo", "primigesta",
              "grande_multipara_v2", "tem_aborto_v2", "aborto_recorrente",
              "sem_filhos_vivos", "perda_previa"]
    _remap = {
        c.lower(): c for c in _MIXED
        if c.lower() in df_mae_enr.columns and c not in df_mae_enr.columns
    }
    if _remap:
        df_mae_enr.rename(columns=_remap, inplace=True)
    df_mae_enr = df_mae_enr.drop_duplicates(subset=["cd_atend_mae_key"], keep="first")
    print(f"gestante_enriquecido loaded: {len(df_mae_enr):,}")
else:
    _con.close()
    print("[07.1] gestante_enriquecido not found — falling back to enrich_obstetric")
    from build_obstetric_features import enrich_obstetric
    LLM_PAR = ROOT / "out" / "llm_extracted_20k.parquet"
    df_mae_enr = enrich_obstetric(
        df_mae_raw.copy().assign(cd_atendimento=df_mae_raw["cd_atend_mae_key"]),
        llm_parquet=str(LLM_PAR) if LLM_PAR.exists() else None,
    )
    df_mae_enr.columns = df_mae_enr.columns.str.lower()
    df_mae_enr = df_mae_enr.drop_duplicates(subset=["cd_atend_mae_key"], keep="first")
    print(f"enrich_obstetric (fallback) done: {len(df_mae_enr):,}")

## 7. Merge all sources

In [ ]:
# Primary merge: Oracle RN + maternal base
df = df_rn_oracle.merge(
    df_mae_raw[[
        "cd_atend_mae_key", "cid_mae", "nm_mae", "idade_mae",
        "ig_mae_struct", "escore_triagem_mae", "sn_cesario_tiss",
        "ds_queixa_principal", "txt_sinais_sintomas", "txt_evolucao_medica"
    ]],
    left_on="cd_atend_mae",
    right_on="cd_atend_mae_key",
    how="left"
)
print(f"[Merge 1] Oracle + mae_raw     : {len(df):,}")

# Enriched obstetric columns merge
OBS_COLS = [
    "cd_atend_mae_key", "G_obs", "A_obs", "C_obs", "N_obs", "IG_obs",
    "tipo_gestacao", "perfil_historico", "risco_gestacao",
    "gestacao_gemelar", "gestacao_pretermo", "primigesta",
    "grande_multipara_v2", "tem_aborto_v2", "aborto_recorrente",
    "sem_filhos_vivos", "perda_previa", "cd_cid"
]
OBS_COLS = [c for c in OBS_COLS if c in df_mae_enr.columns]

df = df.merge(
    df_mae_enr[OBS_COLS],
    left_on="cd_atend_mae",
    right_on="cd_atend_mae_key",
    how="left",
    suffixes=("", "_enr")
)
print(f"[Merge 2] + mae_enr            : {len(df):,}")

# Add internation outcome from Fonte 2 (aggregated by maternal admission)
df_rn_atend["obito_internacao"] = (df_rn_atend["obito_rn"].fillna("N") == "S").astype(int)
df_rn_agg = (
    df_rn_atend.groupby("cd_atend_mae", as_index=False)
    .agg(
        obito_internacao=("obito_internacao", "max"),
        cid_rn_intern=("cid_rn", "first"),
        dt_internacao_rn=("dt_internacao_rn", "min")
    )
)
df = df.merge(df_rn_agg, on="cd_atend_mae", how="left")
df["obito_internacao"] = df["obito_internacao"].fillna(0).astype(int)
print(f"[Merge 3] + rn_atend obitos    : {len(df):,}")

# Gestational age: prefer LLM/text-extracted, fall back to structured
if "ig_obs" in df.columns:
    df["ig_mae"] = df["ig_obs"].fillna(
        df.get("ig_semanas_mae", pd.Series(dtype=float))
    )
else:
    df["ig_mae"] = pd.to_numeric(df.get("ig_semanas_mae", None), errors="coerce")

df.columns = df.columns.str.lower()

## 8. Feature Engineering

In [ ]:
# Numeric coercion
df["vl_peso"]          = pd.to_numeric(df["vl_peso"],          errors="coerce")
df["vl_apgar_5minuto"] = pd.to_numeric(df["vl_apgar_5minuto"], errors="coerce")
df["vl_apgar_1minuto"] = pd.to_numeric(df["vl_apgar_1minuto"], errors="coerce")
df["ig_mae"]           = pd.to_numeric(df["ig_mae"],            errors="coerce")
df["hr_nascimento"]    = pd.to_datetime(df["hr_nascimento"],    errors="coerce")

# Rename for clarity
df = df.rename(columns={
    "vl_peso":           "peso_g",
    "vl_apgar_1minuto":  "apgar_1min",
    "vl_apgar_5minuto":  "apgar_5min",
    "tp_obito_neonatal": "obito_parto_flag",
})

# Fix 3 — obito_rn operator precedence bug (original: | without parens around ==)
df["obito_rn"] = (
    (df["obito_parto_flag"].notna()) | (df["obito_internacao"] == 1)
).astype(int)

df["vinculado_mae"] = df["cd_atend_mae"].notna().astype(int)
df["baixo_peso"]    = (df["peso_g"] < 2500).astype(float)
df["muito_baixo"]   = (df["peso_g"] < 1500).astype(float)
df["micro"]         = (df["peso_g"] < 1000).astype(float)
df["macrossomia"]   = (df["peso_g"] >= 4000).astype(float)
df["apgar_dep"]     = (df["apgar_5min"] < 7).astype(float)
df["prematuro"]     = (df["ig_mae"] < 37).astype(float)
df["mt_prematuro"]  = (df["ig_mae"] < 32).astype(float)
df["cesario"]       = (df["sn_cesario_tiss"].fillna("N") == "S").astype(float)
df["nascido_vivo"]  = df["cd_declaracao_nascido_vivo"].notna().astype(int)

# Maternal CID segment
def _seg(c):
    if pd.isna(c): return "SEM_CID"
    c = str(c).upper()
    if c[:3] in [f"O{i:02d}" for i in range(10, 17)]: return "HIPERTENSIVOS"
    if c[:3] == "O24": return "DIABETES_GEST"
    if c[:3] in ["O60", "O42", "O47"]: return "PARTO_PREMATURO"
    if c[0] == "O" and c[1] in "234": return "HEMORRAGIA"
    if c[:3] in ["O98", "O99"]: return "COMORBIDADES"
    if c[0] == "O": return "OUTROS_OBST"
    return "OUTROS"

df["segmento_mae"] = df["cid_mae"].apply(_seg)

df["faixa_peso"] = pd.cut(
    df["peso_g"],
    bins=[0, 1000, 1500, 2500, 4000, 10000],
    labels=["<1kg", "1-1.5kg", "1.5-2.5kg", "2.5-4kg", ">4kg"],
    right=False
)

# Fix 4 — Restore mixed-case enriched column names after str.lower()
_MIXED = [
    "G_obs", "A_obs", "C_obs", "N_obs", "IG_obs", "tipo_gestacao",
    "perfil_historico", "risco_gestacao", "gestacao_gemelar", "gestacao_pretermo",
    "primigesta", "grande_multipara_v2", "tem_aborto_v2", "aborto_recorrente",
    "sem_filhos_vivos", "perda_previa"
]
_remap = {c.lower(): c for c in _MIXED if c.lower() in df.columns and c not in df.columns}
if _remap:
    df.rename(columns=_remap, inplace=True)

# Flag PE for causal studies
df["T_pe"] = df["cid_mae"].fillna("").str.match(r"^O1[0-6]").astype(int)

# Year of birth (from Oracle hr_nascimento or dt_atendimento)
if "hr_nascimento" in df.columns and df["hr_nascimento"].notna().sum() > 0:
    df["ano_nascimento"] = df["hr_nascimento"].dt.year
elif "dt_atendimento" in df.columns:
    df["ano_nascimento"] = pd.to_datetime(df["dt_atendimento"], errors="coerce").dt.year
else:
    df["ano_nascimento"] = np.nan

print(f"\nDataset final (Fonte 1 + enriquecimento materno):")
print(f"  Total bebes               : {len(df):,}")

## 9. Table 1

In [ ]:
def _pct(s):
    v = s.fillna(0)
    return f"{int(v.sum()):,} ({v.mean()*100:.1f}%)"

def _miq(s):
    s = s.dropna()
    if len(s) == 0: return "-"
    q1, q3 = s.quantile([.25, .75])
    return f"{s.median():.1f} [{q1:.1f}-{q3:.1f}]"

t1 = {
    "N total RN":                   f"{len(df):,}",
    "Vinculados a mae":             _pct(df["vinculado_mae"].astype(float)),
    "-- Desfechos --":              "",
    "  Obito neonatal":             _pct(df["obito_rn"].astype(float)),
    "  Apgar 5min < 7":             _pct(df["apgar_dep"]),
    "  Baixo peso (<2.500g)":       _pct(df["baixo_peso"]),
    "  Muito baixo peso (<1.500g)": _pct(df["muito_baixo"]),
    "  Macrossomia (>4.000g)":      _pct(df["macrossomia"]),
    "-- Biometria --":              "",
    "  Peso (g)":                   _miq(df["peso_g"]),
    "  Apgar 1min":                 _miq(df["apgar_1min"]),
    "  Apgar 5min":                 _miq(df["apgar_5min"]),
    "-- Prematuridade --":          "",
    "  Prematuro (<37s)":           _pct(df["prematuro"]),
    "  Muito prematuro (<32s)":     _pct(df["mt_prematuro"]),
    "-- Perfil materno --":         "",
    "  Mae hipertensiva":           _pct(df["T_pe"].astype(float)),
}
for col, lbl in [
    ("tem_aborto_v2",       "Mae c/ aborto ant. (A>=1)"),
    ("grande_multipara_v2", "Mae grande multipara (G>=5)"),
    ("gestacao_gemelar",    "Gestacao gemelar"),
]:
    if col in df.columns:
        t1[f"  {lbl}"] = _pct(pd.to_numeric(df[col], errors="coerce").fillna(0).astype(float))

tab1 = pd.DataFrame(list(t1.items()), columns=["Caracteristica", "Valor"])
print("\nTabela 1 - Recem-Nascidos:")
print(tab1.to_string(index=False))
tab1.to_csv(OUT_DIR / "tabela1_rn.csv", index=False)

## 10. Quality Report

In [ ]:
print("\n[07.1] Quality Report:")
print(f"  Com peso                  : {df['peso_g'].notna().sum():,} ({df['peso_g'].notna().mean()*100:.1f}%)")
print(f"  Com Apgar 5min            : {df['apgar_5min'].notna().sum():,}")
print(f"  Com IG materna            : {df['ig_mae'].notna().sum():,}")
print(f"  Baixo peso (<2.500g)      : {df['baixo_peso'].sum():,.0f} ({df['baixo_peso'].mean()*100:.1f}%)")
print(f"  Apgar deprimido (<7)      : {df['apgar_dep'].sum():,.0f} ({df['apgar_dep'].mean()*100:.1f}%)")
print(f"  Obito no parto            : {(df['obito_parto_flag'].notna()).sum():,}")
print(f"  Obito na internacao       : {df['obito_internacao'].sum():,}")
print(f"  Obito combinado           : {df['obito_rn'].sum():,}")
print(f"  Com perfil G/A/N materno  : {df['perfil_historico'].notna().sum() if 'perfil_historico' in df.columns else 0:,}")
print(f"  Com segmento_cid materno  : {df['segmento_cid'].notna().sum() if 'segmento_cid' in df.columns else 0:,}")
print(f"  Com ano nascimento        : {df['ano_nascimento'].notna().sum():,}")

## 11. Save Artifacts

In [ ]:
with open(ART_DIR / "rn_dataset.pkl", "wb") as f:
    pickle.dump({"df": df, "df_rn_atend": df_rn_atend}, f)

print(f"\n[07.1] Artefatos salvos: {ART_DIR}")
print(f"[07.1] Tabela 1 salva  : {OUT_DIR / 'tabela1_rn.csv'}")

In [ ]:
print("\n" + "="*60)
print("  07.1 CARGA RN — CONCLUIDO")
print("="*60)
print(f"  Total bebes    : {len(df):,}")
print(f"  Total RN atend : {len(df_rn_atend):,}")
print(f"  Artefatos em   : {ART_DIR}")
print(f"  Tabela 1 em    : {OUT_DIR}")
print("="*60)